In [ ]:
# import sys
# if r"D:\HST\QuantStudio" not in sys.path: sys.path.insert(0, r"D:\HST\QuantStudio")
import logging
import warnings
warnings.filterwarnings('ignore')

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [2]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

# BaoStockDB

`BaoStockDB` 是基于 [BaoStock](http://baostock.com/baostock/) 在线 API 构建的因子库，主要用于**演示和测试**。通过 HTTP 请求实时获取数据，无需本地数据库。

> 注意：BaoStockDB 仅实现了 BaoStock API 的部分接口，不适合生产环境使用。可用于快速验证因子逻辑或编写示例代码。

## 连接参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `UserID` | `"anonymous"` | BaoStock 登录用户 ID |
| `Pwd` | `"123456"` | BaoStock 登录密码 |

通常无需配置文件，使用默认匿名账号即可。如需自定义，配置文件为 `~/QuantStudioConfig/BaoStockDBConfig.json`。

## 因子表类型

BaoStockDB 支持 6 种因子表类型，对应 BaoStock 的不同 API 模式：

| 类型 | 参数模式 | 说明 |
|------|----------|------|
| `DTRangeTable` | `start_date` + `end_date` | 时间区间查询，如 K 线、业绩快报 |
| `DTTable` | `date` | 逐时点查询，如行业分类、成分股、复权因子 |
| `QuarterTable` | `year` + `quarter` | 按季度查询，如季频财务指标 |
| `YearTable` | `year` | 按年份查询，如除权除息信息 |
| `MacroDataTable` | `start_date` + `end_date` | 宏观经济数据，无证券代码维度 |
| `StockBasicTable` | `code` | 证券基本资料，无日期参数 |

## 全部因子表（24 张）

| 表名 | 类型 | 说明 |
|------|------|------|
| A股K线数据 | DTRangeTable | 日/周/月 K 线，含估值指标 |
| 季度业绩快报 | DTRangeTable | 季度业绩快报数据 |
| 季度业绩预告 | DTRangeTable | 季度业绩预告数据 |
| 行业分类 | DTTable | 证监会行业分类 |
| 每日A股K线 | DTTable | 单日 A 股 K 线查询 |
| 每日ETF K线 | DTTable | 单日 ETF K 线查询 |
| 每日复权因子 | DTTable | 前/后复权因子 |
| 每日全部A股 | DTTable | 某日全部在市A股列表 |
| 上证50成分股 | DTTable | 上证 50 成分股列表 |
| 沪深300成分股 | DTTable | 沪深 300 成分股列表 |
| 中证500成分股 | DTTable | 中证 500 成分股列表 |
| 季度盈利能力 | QuarterTable | ROE、净利率、毛利率等 |
| 季度营运能力 | QuarterTable | 应收/存货/资产周转率等 |
| 季度成长能力 | QuarterTable | 净利润/净资产/每股收益同比增长 |
| 季度偿债能力 | QuarterTable | 流动/速动比率、资产负债率等 |
| 季度现金流量 | QuarterTable | 经营现金流相关指标 |
| 季度杜邦指数 | QuarterTable | 杜邦分析分解指标 |
| 除权除息信息 | YearTable | 分红送股除权除息明细 |
| 存款利率 | MacroDataTable | 央行存款基准利率 |
| 贷款利率 | MacroDataTable | 央行贷款基准利率 |
| 存款准备金率 | MacroDataTable | 存款准备金率调整 |
| 货币供应量 | MacroDataTable | M0/M1/M2 月度数据 |
| 货币供应量(年底余额) | MacroDataTable | M0/M1/M2 年底余额 |
| 证券基本资料 | StockBasicTable | 股票代码/名称/上市日期/状态 |

In [3]:
# 创建因子库对象并 connect（需要联网）
from QuantStudio.Factor.BaoStockDB import BaoStockDB

FDB = BaoStockDB().connect()
print(qs_help(FDB))

2026-09-24 15:41:45,727 | QS | 36296 | WARNING : 数据库信息文件: 'D:\HST\QuantStudio\QuantStudio\Resource\BaoStockDBInfo.xlsx' 有更新, 尝试从中导入新信息.


login success!
类型: BaoStockDB
模块: QuantStudio.Factor.BaoStockDB
QS 对象类型: 因子库
QS 对象名称: BaoStockDB
QSID: c71ceadddf186b11a604aabc571d7e357ec1502375d96d737205cb2cc8c5dc9d
参数集:
    * Name(名称): <class 'str'>, 默认值 'BaoStockDB', 当前取值: 'BaoStockDB'
说明文档:
    基于 BaoStock 的因子库
    API: http://baostock.com/baostock/
    库配置信息文件在 QuantStudio 包目录下 Resource 目录下的 BaoStockDBInfo.xlsx, 记录了相关配置信息


In [5]:
# 可用的因子表（24 张）
print(FDB.TableNames[:5])

['A股K线数据', '行业分类', '季度盈利能力', '除权除息信息', '季度营运能力']


# 时点与 ID 获取

BaoStockDB 提供了基本的交易日和股票代码查询。

In [5]:
# 获取交易日序列
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 20))
print(DTs[:5])

login success!
[Timestamp('2022-01-04 00:00:00'), Timestamp('2022-01-05 00:00:00'), Timestamp('2022-01-06 00:00:00'), Timestamp('2022-01-07 00:00:00'), Timestamp('2022-01-10 00:00:00')]


In [6]:
# 获取当前在市的全体 A 股
IDs = FDB.getStockID()
print(IDs[:5])

login success!
['000001.SZ', '000002.SZ', '000006.SZ', '000007.SZ', '000008.SZ']


# DTRangeTable — A股K线数据

`DTRangeTable` 按时间区间查询，一次 API 请求获取整个日期范围的数据。适用于 K 线、业绩快报、业绩预告等。

核心参数：
- `LookBack`：缺失填充回溯天数（默认 0）
- `APIArgs`：透传给 BaoStock API 的额外参数（如 `frequency`、`adjustflag`）

In [7]:
# 获取因子表 — A股K线数据（DTRangeTable）
FT = FDB.getTable("A股K线数据", args={"LookBack": 0})
print(qs_help(FT))

类型: _DTRangeTable
模块: QuantStudio.Factor.BaoStockDB
QS 对象类型: 计算节点-因子表
QS 对象名称: A股K线数据
QSID: 271ac426bc62996304149496cab18ce0cf189810dc9a52b059b492c0998f0384
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'A股K线数据'
    * TableType(因子表类型): typing.Literal['DTRangeTable'], 默认值 'DTRangeTable', 当前取值: 'DTRangeTable'
    * LookBack(回溯天数): <class 'int'>, 默认值 0, 当前取值: 0
说明文档:
    BaoStockDB 库中基于取时间区间数据 API 的因子表


In [8]:
# 因子列表
print(FT.FactorNames)

['open', 'high', 'low', 'close', 'preclose', 'volume', 'amount', 'adjustflag', 'turn', 'tradestatus', 'pctChg', 'peTTM', 'pbMRQ', 'psTTM', 'pcfNcfTTM', 'isST']


In [9]:
# 读取因子表数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据 (Panel):")
print(Data)

因子表数据 (Panel):
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: open to close
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000002.SZ


In [ ]:
# DTRangeTable 无 Field 参数示例：季度业绩快报
# 与 A股K线数据 不同，此表的 ArgInfo 中没有 Field 类型参数，不传 fields 给 API
FT = FDB.getTable("季度业绩快报")
print(f"因子: {FT.FactorNames}")

DTs = [dt.datetime(2024, 6, 30)]
Data = FT.readData(factor_names=["performanceExpressTotalAsset", "performanceExpressROEWa"], ids=["sh.600000"], dts=DTs)
print(Data)

# DTTable — 行业分类 / 成分股 / 复权因子

`DTTable` 逐时点查询，每个日期发一次 API 请求。适用于行业分类、成分股、复权因子、单日 K 线等。

In [ ]:
# DTTable 示例：行业分类
FT = FDB.getTable("行业分类")

DTs = [dt.datetime(2024, 6, 28)]
Data = FT.readData(factor_names=["industry", "industryClassification"], ids=["sh.600000", "sz.000001"], dts=DTs)
print(Data)

# QuarterTable

In [ ]:
# QuarterTable 示例：季度盈利能力
FT = FDB.getTable("季度盈利能力")

DTs = [dt.datetime(2024, 3, 31), dt.datetime(2024, 6, 30), dt.datetime(2024, 9, 30)]
Data = FT.readData(factor_names=["roeAvg", "npMargin", "epsTTM"], ids=["sh.600000"], dts=DTs)
print(Data)

# YearTable

In [ ]:
# YearTable 示例：除权除息信息
FT = FDB.getTable("除权除息信息")
print(f"表类型: {type(FT).__name__}, 因子: {FT.FactorNames}")

DTs = [dt.datetime(2024, 6, 30)]
Data = FT.readData(factor_names=["dividCashPsBeforeTax", "dividStocksPs"], ids=["sh.600000"], dts=DTs)
print(Data)

# MacroDataTable

In [ ]:
# MacroDataTable 示例：存款利率
# MacroDataTable 无证券代码维度，ids 传入占位即可
FT = FDB.getTable("存款利率")

DTs = [dt.datetime(2015, 3, 1), dt.datetime(2015, 10, 24)]
Data = FT.readData(factor_names=["demandDepositRate", "fixedDepositRate1Year"], ids=["ALL"], dts=DTs)
print(Data)

In [8]:
# MacroDataTable 特殊情况：货币供应量（无 Date 字段）
# 当 MacroDataTable 的 FactorInfo 中没有 Date 类型字段时，使用 statYear + statMonth 构造日期
FT = FDB.getTable("货币供应量")
print(f"表类型: {type(FT).__name__}, 因子: {FT.FactorNames}")

DTs = [dt.datetime(2010, 1, 31), dt.datetime(2010, 3, 31)]
Data = FT.readData(factor_names=["m0Month", "m1Month", "m2Month"], ids=["ALL"], dts=DTs)
print(Data.iloc[:, :, 0])

表类型: _MacroDataTable, 因子: ['statYear', 'statMonth', 'm0Month', 'm0YOY', 'm0ChainRelative', 'm1Month', 'm1YOY', 'm1ChainRelative', 'm2Month', 'm2YOY', 'm2ChainRelative']
                         m0Month      m1Month      m2Month
2010-01-31 00:00:00  40758.58000  229588.9800  625609.2900
2010-03-31 00:00:00  39080.58192  229397.9332  649947.4643


In [9]:
# MacroDataTable 特殊情况：货币供应量(年底余额)（无 Date 字段，且无 statMonth 字段）
# 当 MacroDataTable 的 FactorInfo 中没有 Date 类型字段时，使用 statYear 构造日期
FT = FDB.getTable("货币供应量(年底余额)")
print(f"表类型: {type(FT).__name__}, 因子: {FT.FactorNames}")

DTs = [dt.datetime(2010, 12, 31), dt.datetime(2011, 12, 31)]
Data = FT.readData(factor_names=["m0Year", "m0YearYOY", "m1Year"], ids=["ALL"], dts=DTs)
print(Data.iloc[:, :, 0])

表类型: _MacroDataTable, 因子: ['statYear', 'm0Year', 'm0YearYOY', 'm1Year', 'm1YearYOY', 'm2Year', 'm2YearYOY']
                       m0Year  m0YearYOY    m1Year
2010-12-31 00:00:00  44628.20       16.7  266621.5
2011-12-31 00:00:00  50748.46       13.8  289847.7


# StockBasicTable

In [ ]:
# StockBasicTable 示例：证券基本资料
# StockBasicTable 无日期参数，dts 传入占位即可
FT = FDB.getTable("证券基本资料")
print(f"表类型: {type(FT).__name__}, 因子: {FT.FactorNames}")

Data = FT.readData(factor_names=["code_name", "ipoDate", "status"], ids=["sh.600000", "sz.000001"], dts=[dt.datetime(2024, 12, 31)])
print(Data)

# 因子

通过 `FT.getFactor()` 获取的因子对象与其他因子库中的因子完全一致，同样支持运算符重载和衍生因子。详见 **[基本框架](基本框架.ipynb)** 和 **[因子开发](因子开发.ipynb)**。